In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# LENDO OS DADOS DA CAMADA SILVER
df_manobras = spark.table("workspace.default.silver_manobras")
df_alarmes = spark.table("workspace.default.silver_alarmes")

# ======================================================================================
# PREPARANDO OS PERÍODOS DE MANOBRAS
# ======================================================================================
# OBJETIVO: Para entender a causa raiz dos alarmes de velocidade (se foi erro do 
# comandante ou exigência da operação), precisamos descobrir o status exato do 
# rebocador no momento do disparo.

# Fase 1: Deslocando
manobras_deslocando = df_manobras.select(
    F.col("REBOCADOR"), F.col("PORTO"),
    F.col("DT_INI_DESLOC").alias("start_time"),
    F.col("DT_INI_OPER").alias("end_time"),
    F.lit("Deslocando para a manobra").alias("Status_Calculado"),
    F.col("ID_EVENTO"), F.col("NAVIO")
)

# Fase 2: Operando
manobras_operando = df_manobras.select(
    F.col("REBOCADOR"), F.col("PORTO"),
    F.col("DT_INI_OPER").alias("start_time"),
    F.col("DT_FIM_OPER").alias("end_time"),
    F.lit("Operando").alias("Status_Calculado"),
    F.col("ID_EVENTO"), F.col("NAVIO")
)

# Fase 3: Retornando
manobras_retornando = df_manobras.select(
    F.col("REBOCADOR"), F.col("PORTO"),
    F.col("DT_FIM_OPER").alias("start_time"),
    F.col("DT_FIM_DESLOC").alias("end_time"),
    F.lit("Retornando da manobra").alias("Status_Calculado"),
    F.col("ID_EVENTO"), F.col("NAVIO")
)

# Fase 4: Atender Sequência (Intervalos de até 30 min)
window_spec = Window.partitionBy("REBOCADOR", "PORTO").orderBy("DT_INI_DESLOC")
df_manobras_seq = df_manobras.withColumn("prox_inicio_desloc", F.lead("DT_INI_DESLOC", 1).over(window_spec))

intervalos = df_manobras_seq.withColumn(
    "intervalo_segundos",
    F.col("prox_inicio_desloc").cast("long") - F.col("DT_FIM_DESLOC").cast("long")
)

manobras_sequencia = intervalos.filter(
    (F.col("intervalo_segundos") > 0) & (F.col("intervalo_segundos") <= 1800)
).select(
    F.col("REBOCADOR"), F.col("PORTO"),
    F.col("DT_FIM_DESLOC").alias("start_time"),
    F.col("prox_inicio_desloc").alias("end_time"),
    F.lit("Atender sequência").alias("Status_Calculado"),
    F.col("ID_EVENTO"), F.col("NAVIO")
)

# Unificando todos os períodos
periodos_com_status = manobras_deslocando.unionByName(manobras_operando) \
    .unionByName(manobras_retornando) \
    .unionByName(manobras_sequencia)

# ==========================================
# CRUZANDO ALARMES COM MANOBRAS
# ==========================================

# Adiciona um ID único aos alarmes
df_alarmes_with_id = df_alarmes.withColumn("alarm_id", F.monotonically_increasing_id())

# O Join: Verifica se o disparo do alarme aconteceu DENTRO do período
alarmes_joined = df_alarmes_with_id.alias("a").join(
    periodos_com_status.alias("p"),
    (F.col("a.Embarcação") == F.col("p.REBOCADOR")) &
    (F.col("a.Localização") == F.col("p.PORTO")) &
    (F.col("a.Disparo").between(F.col("p.start_time"), F.col("p.end_time"))),
    "left"
)

# Define prioridades
priority_map = F.create_map(
    F.lit("Operando"), F.lit(1),
    F.lit("Deslocando para a manobra"), F.lit(2),
    F.lit("Retornando da manobra"), F.lit(3),
    F.lit("Atender sequência"), F.lit(4)
)
alarmes_joined = alarmes_joined.withColumn("priority", priority_map[F.col("Status_Calculado")])

window_spec_dedup = Window.partitionBy("alarm_id").orderBy(F.col("priority").asc())

# Limpa o DataFrame final
df_alarmes_final = alarmes_joined.withColumn("row_num", F.row_number().over(window_spec_dedup)) \
    .filter(F.col("row_num") == 1) \
    .withColumn("Status", F.coalesce(F.col("Status_Calculado"), F.lit("Sem manobra"))) \
    .select("a.*", "Status", "p.ID_EVENTO", "p.NAVIO")

# ==========================================
# SALVANDO NA CAMADA GOLD
# ==========================================
df_alarmes_final.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_fato_alarmes")

print("Camada Gold Concluída! Tabela final pronta para análise.")

---

# Analisando o resultado

In [0]:
# Lendo a tabela final da camada Gold
df_gold = spark.table("workspace.default.gold_fato_alarmes")

# Pegando o total geral de alarmes para calcular a porcentagem
total_alarmes = df_gold.count()

# Agrupando por Status, contando e calculando a porcentagem
df_analise = df_gold.groupBy("Status").agg(
    F.count("*").alias("Total_Alarmes")
).withColumn(
    "Porcentagem", 
    F.round((F.col("Total_Alarmes") / total_alarmes) * 100, 2)
).orderBy(F.col("Total_Alarmes").desc())

# Exibindo o resultado
display(df_analise)

Databricks visualization. Run in Databricks to view.

O cruzamento da base de alarmes de velocidade com os registros de manobras nos permitiu categorizar o contexto operacional de cada infração. A partir da visualização gerada, extraímos os seguintes insights para a tomada de decisão da gestão:

- Foco de Atuação (Deslocamentos): Cerca de 30,88% dos alarmes ocorreram em momentos onde o comandante tem total autonomia sobre a embarcação (17,10% deslocando-se para a manobra e 13,78% retornando). Somando a isso os 0,26% em atendimento de sequência, temos um total de 31,14% de casos acionáveis. A recomendação para a gestão é orientar os comandantes a iniciarem o deslocamento com maior antecedência, mitigando a necessidade de exceder o limite de velocidade para cumprir o horário.

- Alarmes Justificáveis (Operação): Identificamos que 22,08% dos disparos ocorreram com o status "Operando". Como a velocidade nestas fases é frequentemente ditada pela inércia do navio assistido ou pelas ordens do prático, estes alarmes fogem do controle direto do comandante do rebocador e, portanto, não devem ser o foco primário de penalizações ou cobranças.

- Oportunidade e Trabalhos Futuros (Sem Manobra): A maior fatia dos registros (46,77%) foi classificada como "Sem manobra". Isto não invalida a análise, mas escancara uma limitação do conjunto de dados atual, que contempla apenas manobras comerciais. Como proposta de trabalhos futuros para enriquecer este MVP, recomenda-se a ingestão de dados de ordens de manutenção, abastecimento e transferências entre portos. Cruzar essas novas fontes permitirá mapear a causa raiz dessa fatia oculta de deslocamentos.

In [0]:
# 2. Pivotando para contar a quantidade absoluta por Status
df_counts = df_gold.groupBy("Localização") \
    .pivot("Status") \
    .agg(F.count("alarm_id")) \
    .na.fill(0)

# 3. Calculando o Total Geral de alarmes de cada Porto
df_counts = df_counts.withColumn(
    "Total_Geral", 
    F.col("Deslocando para a manobra") + F.col("Operando") + 
    F.col("Retornando da manobra") + F.col("Sem manobra") + F.col("Atender sequência")
)

# 4. Convertendo cada status para PORCENTAGEM (%) do total daquele porto
lista_status = ["Deslocando para a manobra", "Operando", "Retornando da manobra", "Sem manobra", "Atender sequência"]

df_portos_pct = df_counts
for st in lista_status:
    df_portos_pct = df_portos_pct.withColumn(
        st, 
        F.round((F.col(st) / F.col("Total_Geral")) * 100, 2)
    )

# Ordena do porto com maior volume total para o menor
df_portos_pct = df_portos_pct.orderBy(F.col("Total_Geral").desc())

# 5. Exibindo a tabela formatada em %
display(df_portos_pct)

Databricks visualization. Run in Databricks to view.

#Análise Regional — Comportamento por Porto
A distribuição relativa por porto expõe dinâmicas operacionais bastante distintas entre os terminais:

- Portos com Maior Deslocamento (Foco de Orientação): Terminais como Porto de Santos/SP e Porto de Salvador/BA apresentam proporções expressivas de alarmes concentradas em Deslocando e Retornando da Manobra (barras azul e verde). Isso indica que, nestas localidades, as distâncias entre os pontos de apoio e os berços exigem atenção redobrada dos comandantes para planejar a navegação sem excesso de velocidade.

- Portos com Predomínio de Operação: Locais com maior fatia na cor vermelha (Operando) refletem manobras mais complexas ou acompanhamento prolongado de navios em canal de acesso, onde a velocidade é imposta por fatores externos.

- Gargalos de Informação (Sem Manobra): Portos com predomínio do status roxo (Sem Manobra) indicam locais onde ocorrem muitas movimentações não comerciais (como rotinas de abastecimento local ou manutenção), reforçando a recomendação de integrar novas fontes de dados no futuro.
